# Brain Tumor MRI Classifier — Kaggle Training Notebook

**Architecture:** EfficientNetB4 (ImageNet pretrained)  
**Strategy:** 5-phase gradual unfreezing with warmup + cosine decay LR  
**Augmentation:** CLAHE + Z-score + Gaussian noise + elastic deformation  
**Imbalance:** Class-weighted loss + label smoothing  

---

### ⚠️ Before Running
1. Make sure you have uploaded your dataset as a Kaggle Dataset  
2. Enable **GPU T4 x2** under Settings → Accelerator  
3. Run all cells top to bottom  

## 1. Setup & Reproducibility

In [ ]:
import os
import sys
import random
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, mixed_precision

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Mixed precision (2–3× faster on Tensor Core GPUs) ──────────────────────
mixed_precision.set_global_policy('mixed_float16')

print(f'TensorFlow version : {tf.__version__}')
print(f'Mixed precision    : {mixed_precision.global_policy().name}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available     : {gpus if gpus else "None — using CPU"}')

## 2. Configuration

**⚠️ UPDATE `DATA_ROOT` below** to point to your uploaded Kaggle dataset.  
Use the "Add Input" button on the right sidebar to attach your dataset, then check the path with `!ls /kaggle/input/`.

In [ ]:
# ── CHECK YOUR DATASET PATH ─────────────────────────────────────────────────
# Run this to see what datasets are available:
!ls /kaggle/input/
print('---')
# Then update DATA_ROOT below with the correct path

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
# ⚠️ CHANGE THIS to match your dataset name on Kaggle:
DATA_ROOT    = Path('/kaggle/input/brain-tumor-mri-dataset')

# Output directories (Kaggle allows writing to /kaggle/working/)
MODELS_DIR   = Path('/kaggle/working/models')
REPORTS_DIR  = Path('/kaggle/working/reports')
LOGS_DIR     = Path('/kaggle/working/logs')

for d in [MODELS_DIR, REPORTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = str(MODELS_DIR / 'best_model.keras')

# ── Model / training hyperparameters ───────────────────────────────────────
IMG_SIZE    = (380, 380)   # EfficientNetB4 native resolution
BATCH_SIZE  = 16
NUM_CLASSES = 4
L2_REG      = 1e-4
DROPOUT_1   = 0.4
DROPOUT_2   = 0.3

CLASS_NAMES = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

# ── Verify data exists ─────────────────────────────────────────────────────
for split in ['train', 'val', 'test']:
    p = DATA_ROOT / split
    if p.exists():
        print(f'  ✅ {split}: {p} ({len(list(p.iterdir()))} class dirs)')
    else:
        print(f'  ❌ {split}: {p} NOT FOUND — check DATA_ROOT!')

print(f'\nConfiguration loaded.')
print(f'  Image size : {IMG_SIZE}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Classes    : {CLASS_NAMES}')

## 3. Data Loading

In [ ]:
def make_raw_dataset(directory, shuffle=False):
    """Load images from directory structure into a tf.data.Dataset."""
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        shuffle=shuffle,
        seed=SEED if shuffle else None,
        class_names=CLASS_NAMES,
    )
    return ds

train_ds_raw = make_raw_dataset(DATA_ROOT / 'train', shuffle=True)
val_ds_raw   = make_raw_dataset(DATA_ROOT / 'val',   shuffle=False)
test_ds_raw  = make_raw_dataset(DATA_ROOT / 'test',  shuffle=False)

print('Class names :', train_ds_raw.class_names)
print(f'Train batches: {len(train_ds_raw)}')
print(f'Val   batches: {len(val_ds_raw)}')
print(f'Test  batches: {len(test_ds_raw)}')

## 4. Class Distribution

In [ ]:
class_counts = {}
for split in ['train', 'val', 'test']:
    split_dir = DATA_ROOT / split
    counts = {}
    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls
        if cls_dir.exists():
            counts[cls] = len(list(cls_dir.glob('*')))
        else:
            counts[cls] = 0
    class_counts[split] = counts
    print(f'  {split:5s}: {counts}  (total: {sum(counts.values())})')

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
for ax, split in zip(axes, ['train', 'val', 'test']):
    counts = class_counts[split]
    ax.bar(counts.keys(), counts.values(), color=colors)
    ax.set_title(f'{split.capitalize()} set')
    ax.set_ylabel('Images')
    ax.tick_params(axis='x', rotation=20)
fig.suptitle('Class Distribution per Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Class Weights (Handle Imbalance)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

all_labels = []
for _, labels in train_ds_raw.unbatch():
    all_labels.append(int(np.argmax(labels.numpy())))
all_labels = np.array(all_labels)

cw_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=all_labels
)
class_weights = {i: float(w) for i, w in enumerate(cw_array)}

print('Class weights:')
for idx, cls in enumerate(CLASS_NAMES):
    print(f'  [{idx}] {cls:<12s}: {class_weights[idx]:.4f}')

## 6. MRI Augmentation Pipeline (Inlined)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Augmentation functions (from utils/augmentation.py — inlined for Kaggle)
# ═══════════════════════════════════════════════════════════════════════════

def apply_clahe(image_np):
    """CLAHE contrast enhancement per channel."""
    img_uint8 = (np.clip(image_np, 0.0, 1.0) * 255).astype(np.uint8)
    if img_uint8.ndim == 2:
        img_uint8 = np.stack([img_uint8] * 3, axis=-1)
    elif img_uint8.ndim == 3 and img_uint8.shape[2] == 1:
        img_uint8 = np.concatenate([img_uint8] * 3, axis=-1)
    elif img_uint8.ndim == 3 and img_uint8.shape[2] != 3:
        img_uint8 = np.stack([img_uint8[:, :, 0]] * 3, axis=-1)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    channels = cv2.split(img_uint8)
    enhanced = cv2.merge([clahe.apply(ch) for ch in channels])
    return (enhanced / 255.0).astype(np.float32)

def zscore_normalize(image_np):
    """Per-image z-score normalization → [0, 1]."""
    mean = image_np.mean()
    std = image_np.std() + 1e-7
    normalized = (image_np - mean) / std
    normalized = np.clip(normalized, -3.0, 3.0)
    return ((normalized + 3.0) / 6.0).astype(np.float32)

def add_gaussian_noise(image_np, std=0.015):
    noise = np.random.normal(0.0, std, image_np.shape).astype(np.float32)
    return np.clip(image_np + noise, 0.0, 1.0)

def apply_random_blur(image_np, max_sigma=0.8):
    sigma = np.random.uniform(0.0, max_sigma)
    if sigma < 0.1:
        return image_np
    img_uint8 = (image_np * 255).astype(np.uint8)
    k = max(3, int(2 * round(2 * sigma) + 1))
    if k % 2 == 0:
        k += 1
    blurred = cv2.GaussianBlur(img_uint8, (k, k), sigma)
    return (blurred / 255.0).astype(np.float32)

def elastic_transform(image_np, alpha=25.0, sigma=4.0):
    h, w = image_np.shape[:2]
    dx = cv2.GaussianBlur((np.random.rand(h, w).astype(np.float32) * 2 - 1), (0, 0), sigma) * alpha
    dy = cv2.GaussianBlur((np.random.rand(h, w).astype(np.float32) * 2 - 1), (0, 0), sigma) * alpha
    x_coords, y_coords = np.meshgrid(np.arange(w), np.arange(h))
    map_x = (x_coords + dx).astype(np.float32)
    map_y = (y_coords + dy).astype(np.float32)
    img_uint8 = (image_np * 255).astype(np.uint8)
    distorted = cv2.remap(img_uint8, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    return (distorted / 255.0).astype(np.float32)

def random_brightness_contrast(image_np, brightness_range=0.15, contrast_range=0.15):
    alpha = 1.0 + np.random.uniform(-contrast_range, contrast_range)
    beta = np.random.uniform(-brightness_range, brightness_range)
    return np.clip(image_np * alpha + beta, 0.0, 1.0).astype(np.float32)

def mri_augment(image_np, training=True):
    """Full MRI preprocessing + augmentation pipeline."""
    if image_np.max() > 1.5:
        image_np = image_np / 255.0
    image_np = image_np.astype(np.float32)
    if image_np.ndim == 2:
        image_np = np.stack([image_np] * 3, axis=-1)
    elif image_np.shape[-1] == 1:
        image_np = np.concatenate([image_np] * 3, axis=-1)
    image_np = apply_clahe(image_np)
    image_np = zscore_normalize(image_np)
    if not training:
        return image_np
    if np.random.rand() < 0.50:
        image_np = add_gaussian_noise(image_np, std=0.015)
    if np.random.rand() < 0.30:
        image_np = apply_random_blur(image_np, max_sigma=0.8)
    if np.random.rand() < 0.40:
        image_np = elastic_transform(image_np, alpha=25.0, sigma=4.0)
    if np.random.rand() < 0.40:
        image_np = random_brightness_contrast(image_np, 0.12, 0.12)
    return image_np

print('Augmentation pipeline defined.')

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def augment_train(image, label):
    """Apply full MRI augmentation pipeline (training mode)."""
    image = tf.py_function(
        lambda img: mri_augment(img.numpy(), training=True),
        [image], tf.float32
    )
    image.set_shape([*IMG_SIZE, 3])
    return image, label

def preprocess_eval(image, label):
    """Apply deterministic preprocessing only (val/test)."""
    image = tf.py_function(
        lambda img: mri_augment(img.numpy(), training=False),
        [image], tf.float32
    )
    image.set_shape([*IMG_SIZE, 3])
    return image, label

# ── Build tf.data pipelines (unbatch → per-image augment → rebatch) ────────
train_ds = (
    train_ds_raw
    .unbatch()
    .cache()
    .map(augment_train, num_parallel_calls=AUTOTUNE)
    .shuffle(buffer_size=1024, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds_raw
    .unbatch()
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_ds_raw
    .unbatch()
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Quick sanity check
for images, labels in train_ds.take(1):
    print(f'Batch shape : {images.shape}  dtype: {images.dtype}')
    print(f'Labels shape: {labels.shape}')
    print(f'Pixel range : [{images.numpy().min():.3f}, {images.numpy().max():.3f}]')

print('Data pipelines ready.')

## 7. Visualise Augmented Samples

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for images, labels in train_ds.take(1):
    for i in range(min(8, len(images))):
        img = images[i].numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        cls_idx = int(np.argmax(labels[i]))
        axes[0, i].imshow(img)
        axes[0, i].set_title(CLASS_NAMES[cls_idx], fontsize=8)
        axes[0, i].axis('off')
        axes[1, i].axis('off')
fig.suptitle('Sample Training Images (after augmentation)', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Model Architecture (EfficientNetB4)

In [ ]:
def build_model():
    inputs = layers.Input(shape=(*IMG_SIZE, 3), name='input_image')
    backbone = tf.keras.applications.EfficientNetB4(
        include_top=False, weights='imagenet',
        input_tensor=inputs, pooling=None,
    )
    backbone.trainable = False
    x = backbone.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(512, kernel_regularizer=regularizers.l2(L2_REG), name='dense_1')(x)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Activation('relu', name='relu_1')(x)
    x = layers.Dropout(DROPOUT_1, name='dropout_1')(x)
    x = layers.Dense(256, kernel_regularizer=regularizers.l2(L2_REG), name='dense_2')(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    x = layers.Activation('relu', name='relu_2')(x)
    x = layers.Dropout(DROPOUT_2, name='dropout_2')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax',
                           kernel_regularizer=regularizers.l2(L2_REG),
                           dtype='float32', name='predictions')(x)
    model = models.Model(inputs=inputs, outputs=outputs, name='BrainTumorClassifier_B4')
    return model, backbone

model, backbone = build_model()
model.summary(show_trainable=True)
print(f'\nTotal params     : {model.count_params():,}')
print(f'Trainable params : {sum(p.numpy().size for p in model.trainable_weights):,}')

## 9. LR Schedule, Callbacks & Compile Helper

In [ ]:
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, total_steps, warmup_steps):
        super().__init__()
        self.base_lr = float(base_lr)
        self.total_steps = int(total_steps)
        self.warmup_steps = int(warmup_steps)
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.base_lr * (step / self.warmup_steps)
        cosine_lr = 0.5 * self.base_lr * (
            1.0 + tf.cos(math.pi * (step - self.warmup_steps) / (self.total_steps - self.warmup_steps))
        )
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {'base_lr': self.base_lr, 'total_steps': self.total_steps, 'warmup_steps': self.warmup_steps}

def make_callbacks(phase_name):
    return [
        tf.keras.callbacks.ModelCheckpoint(filepath=MODEL_SAVE_PATH, monitor='val_accuracy',
                                          save_best_only=True, mode='max', verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=7, min_delta=0.001,
                                        restore_best_weights=True, mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                                            min_lr=1e-7, verbose=1),
        tf.keras.callbacks.CSVLogger(str(LOGS_DIR / 'training_log.csv'), append=True),
    ]

def compile_model(model, base_lr, n_epochs):
    num_train_samples = sum(class_counts['train'].values())
    steps_per_epoch = math.ceil(num_train_samples / BATCH_SIZE)
    total_steps = steps_per_epoch * n_epochs
    warmup_steps = steps_per_epoch
    schedule = WarmupCosineDecay(base_lr, total_steps, warmup_steps)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=schedule),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy'],
    )

print('LR schedule and callback factory ready.')

## 10. Gradual Unfreezing Helper

In [ ]:
def unfreeze_top_fraction(backbone, fraction):
    backbone.trainable = True
    total = len(backbone.layers)
    freeze_until = int(total * (1.0 - fraction))
    for i, layer in enumerate(backbone.layers):
        if i < freeze_until:
            layer.trainable = False
        elif isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True
    trainable_count = sum(1 for l in backbone.layers if l.trainable)
    print(f'  Backbone layers trainable: {trainable_count} / {total}')

print('Gradual unfreezing helper ready.')

## 11. Phase 1 — Head Training (Backbone Frozen)

In [ ]:
PHASE1_EPOCHS = 15
PHASE1_LR     = 1e-3

backbone.trainable = False
compile_model(model, PHASE1_LR, PHASE1_EPOCHS)

print(f'Phase 1: {PHASE1_EPOCHS} epochs, lr={PHASE1_LR}, backbone FROZEN')
history_p1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=make_callbacks('p1'),
    class_weight=class_weights,
)

## 12. Phase 2 — Unfreeze Top 25%

In [ ]:
PHASE2_EPOCHS = 15
PHASE2_LR     = 1e-4

print('Phase 2: unfreeze top 25% of backbone')
unfreeze_top_fraction(backbone, 0.25)
compile_model(model, PHASE2_LR, PHASE2_EPOCHS)

history_p2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=make_callbacks('p2'),
    class_weight=class_weights,
)

## 13. Phase 3 — Unfreeze Top 50%

In [ ]:
PHASE3_EPOCHS = 15
PHASE3_LR     = 5e-5

print('Phase 3: unfreeze top 50% of backbone')
unfreeze_top_fraction(backbone, 0.50)
compile_model(model, PHASE3_LR, PHASE3_EPOCHS)

history_p3 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE3_EPOCHS,
    callbacks=make_callbacks('p3'),
    class_weight=class_weights,
)

## 14. Phase 4 — Unfreeze Top 75%

In [ ]:
PHASE4_EPOCHS = 15
PHASE4_LR     = 2e-5

print('Phase 4: unfreeze top 75% of backbone')
unfreeze_top_fraction(backbone, 0.75)
compile_model(model, PHASE4_LR, PHASE4_EPOCHS)

history_p4 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE4_EPOCHS,
    callbacks=make_callbacks('p4'),
    class_weight=class_weights,
)

## 15. Phase 5 — Full Fine-Tune

In [ ]:
PHASE5_EPOCHS = 10
PHASE5_LR     = 5e-6

print('Phase 5: unfreeze all backbone layers (BN still frozen)')
unfreeze_top_fraction(backbone, 1.0)
compile_model(model, PHASE5_LR, PHASE5_EPOCHS)

history_p5 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE5_EPOCHS,
    callbacks=make_callbacks('p5'),
    class_weight=class_weights,
)

## 16. Load Best Checkpoint

In [ ]:
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
print(f'Best model loaded from: {MODEL_SAVE_PATH}')

## 17. Training History

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Evaluation functions (from utils/evaluation.py — inlined for Kaggle)
# ═══════════════════════════════════════════════════════════════════════════
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

_CLASS_COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

def evaluate_on_split(model, dataset, verbose=True):
    y_true_list, y_pred_list = [], []
    for batch_images, batch_labels in dataset:
        preds = model.predict(batch_images, verbose=0)
        y_pred_list.append(preds)
        y_true_list.append(batch_labels.numpy())
    y_true = np.concatenate(y_true_list, axis=0)
    y_pred_probs = np.concatenate(y_pred_list, axis=0)
    y_true_idx = np.argmax(y_true, axis=1)
    y_pred_idx = np.argmax(y_pred_probs, axis=1)
    accuracy = np.mean(y_true_idx == y_pred_idx)
    if verbose:
        print(f'  Accuracy: {accuracy*100:.2f}%  ({int(accuracy*len(y_true_idx))}/{len(y_true_idx)} correct)')
    return y_true_idx, y_pred_idx, y_pred_probs

def plot_confusion_matrix(y_true, y_pred, class_names=CLASS_NAMES, split='test'):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Confusion Matrix — {split} set', fontsize=14, fontweight='bold')
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0], linewidths=0.5)
    axes[0].set_title('Raw Counts'); axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges', xticklabels=class_names, yticklabels=class_names, ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
    axes[1].set_title('Row-Normalized'); axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / f'{split}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

def plot_roc_curves(y_true_idx, y_pred_probs, class_names=CLASS_NAMES, split='test'):
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true_idx, classes=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(10, 8))
    auc_values = []
    for i, (cls_name, color) in enumerate(zip(class_names, _CLASS_COLORS)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
        roc_auc = auc(fpr, tpr)
        auc_values.append(roc_auc)
        ax.plot(fpr, tpr, color=color, lw=2.0, label=f'{cls_name} (AUC={roc_auc:.3f})')
    macro_auc = float(np.mean(auc_values))
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.6, label='Random (AUC=0.500)')
    ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {split} | Macro AUC={macro_auc:.3f}', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / f'{split}_roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    return macro_auc

def plot_training_history(histories):
    phase_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Training History Across All Phases', fontsize=14, fontweight='bold')
    epoch_offset = 0
    for (phase_name, history), color in zip(histories.items(), phase_colors):
        hist = history.history
        n_epochs = len(hist['accuracy'])
        epochs = range(epoch_offset + 1, epoch_offset + n_epochs + 1)
        axes[0].plot(epochs, hist['accuracy'], color=color, lw=2, label=f'{phase_name} (train)')
        axes[0].plot(epochs, hist['val_accuracy'], color=color, lw=2, ls='--', label=f'{phase_name} (val)')
        axes[1].plot(epochs, hist['loss'], color=color, lw=2, label=f'{phase_name} (train)')
        axes[1].plot(epochs, hist['val_loss'], color=color, lw=2, ls='--', label=f'{phase_name} (val)')
        if epoch_offset > 0:
            axes[0].axvline(epoch_offset+1, color='gray', lw=0.8, ls=':')
            axes[1].axvline(epoch_offset+1, color='gray', lw=0.8, ls=':')
        epoch_offset += n_epochs
    axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(fontsize=7, ncol=2); axes[0].grid(True, alpha=0.3); axes[0].set_ylim([0,1.05])
    axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=7, ncol=2); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Evaluation functions defined.')

In [ ]:
all_histories = {
    'Phase 1 (frozen)' : history_p1,
    'Phase 2 (top 25%)': history_p2,
    'Phase 3 (top 50%)': history_p3,
    'Phase 4 (top 75%)': history_p4,
    'Phase 5 (full)'   : history_p5,
}
plot_training_history(all_histories)

## 18. Evaluation on Validation Set

In [ ]:
print('Evaluating on VALIDATION set...')
y_true_val, y_pred_val, y_probs_val = evaluate_on_split(best_model, val_ds)
plot_confusion_matrix(y_true_val, y_pred_val, split='val')
val_auc = plot_roc_curves(y_true_val, y_probs_val, split='val')
print(classification_report(y_true_val, y_pred_val, target_names=CLASS_NAMES, digits=4))

## 19. Final Evaluation on Test Set

In [ ]:
print('Evaluating on TEST set...')
y_true_test, y_pred_test, y_probs_test = evaluate_on_split(best_model, test_ds)
plot_confusion_matrix(y_true_test, y_pred_test, split='test')
test_auc = plot_roc_curves(y_true_test, y_probs_test, split='test')
print(classification_report(y_true_test, y_pred_test, target_names=CLASS_NAMES, digits=4))

## 20. Save Model & Metadata

In [ ]:
# Save label map
label_map = {name: i for i, name in enumerate(CLASS_NAMES)}
with open(MODELS_DIR / 'label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)
print(f'Saved: {MODELS_DIR / "label_map.json"}')

# Save model info
model_info = {
    'architecture': 'EfficientNetB4',
    'input_size': list(IMG_SIZE) + [3],
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'training_phases': 5,
    'total_epochs': 70,
}
with open(MODELS_DIR / 'model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print(f'Saved: {MODELS_DIR / "model_info.json"}')

print('\n✅ Training complete! Download best_model.keras from /kaggle/working/models/')

## 21. Download Instructions

After training completes:
1. Click **"Save Version"** (top right) → **Save & Run All** → **Quick Save**
2. Go to the **Output** tab of your notebook
3. Download these files to your local project's `models/` folder:
   - `models/best_model.keras`
   - `models/label_map.json`
   - `models/model_info.json`
4. Also download everything from `reports/` for your documentation